### Function

In [2]:
import re
from pathlib import Path

import pandas as pd


# =========================
# Config
# =========================
def parse_results(log_path: str):
    output_txt_path = "best_results_by_shared_parameter_set.txt"
    output_csv_path = "best_after_test_real_mae_per_shared_param_set_horizon.csv"
    avg_csv_path = "avg_after_test_real_mae_by_shared_param_set.csv"


    # =========================
    # Helpers
    # =========================

    def parse_value(value: str):
        """
        Convert string values from the log into Python types.
        Handles:
        - true / false
        - ints
        - floats
        - percentages, e.g. 35.889217%
        - strings
        """
        value = value.strip()

        if value.lower() == "true":
            return True
        if value.lower() == "false":
            return False

        # Strip percentage signs if present
        value_no_pct = value.rstrip("%")

        try:
            if "." in value_no_pct or "e" in value_no_pct.lower():
                return float(value_no_pct)
            return int(value_no_pct)
        except ValueError:
            return value


    def parse_random_search_log(log_path: str) -> pd.DataFrame:
        """
        Parse random-search log into a candidate-level DataFrame.

        Expected structure:
        Shared parameter set X: Used params | ...
        Model horizon Y:
        tuning_batch_size A/B | candidate_index=... | ... | after_test_real_mae=...
        """
        rows = []

        current_set = None
        current_set_params = None
        current_horizon = None

        with open(log_path, "r", encoding="utf-8") as f:
            for raw_line in f:
                line = raw_line.strip()

                if not line:
                    continue

                # Example:
                # Shared parameter set 1: Used params | text_attn_layers=[1, 2] | ...
                m_set = re.match(r"^Shared parameter set\s+(\d+):\s*(.*)$", line)
                if m_set:
                    current_set = int(m_set.group(1))
                    current_set_params = m_set.group(2)
                    current_horizon = None
                    continue

                # Skip Shared Baseline section before the first Shared parameter set
                if current_set is None:
                    continue

                # Example:
                # Model horizon 1:
                m_horizon = re.match(r"^Model horizon\s+(\d+):", line)
                if m_horizon:
                    current_horizon = int(m_horizon.group(1))
                    continue

                # Candidate rows start with tuning_batch_size
                if not line.startswith("tuning_batch_size"):
                    continue

                if current_horizon is None:
                    continue

                parts = [p.strip() for p in line.split("|")]

                row = {
                    "shared_param_set": current_set,
                    "shared_param_params": current_set_params,
                    "horizon": current_horizon,
                }

                # Parse first segment:
                # tuning_batch_size 1/8
                m_batch = re.match(r"tuning_batch_size\s+(\d+)/(\d+)", parts[0])
                if m_batch:
                    row["tuning_batch_size_index"] = int(m_batch.group(1))
                    row["tuning_batch_size"] = int(m_batch.group(2))

                # Parse key=value fields
                for part in parts[1:]:
                    if "=" not in part:
                        continue

                    key, value = part.split("=", 1)
                    row[key.strip()] = parse_value(value)

                rows.append(row)

        return pd.DataFrame(rows)


    # =========================
    # Parse log
    # =========================

    df = parse_random_search_log(log_path)

    required_cols = [
        "shared_param_set",
        "horizon",
        "candidate_index",
        "tuning_batch_size",
        "best_epoch",
        "before_val_real_mae",
        "after_val_real_mae",
        "before_test_real_mae",
        "after_test_real_mae",
        "shared_param_params",
    ]

    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing expected columns: {missing_cols}")

    print(f"Parsed candidate rows: {len(df)}")
    print(f"Shared parameter sets found: {df['shared_param_set'].nunique()}")
    print(f"Horizons found: {sorted(df['horizon'].unique())}")


    # =========================
    # Find best after_test_real_mae per shared parameter set + horizon
    # =========================

    best_idx = (
        df.groupby(["shared_param_set", "horizon"])["after_test_real_mae"]
        .idxmin()
    )

    best_per_set_horizon = (
        df.loc[best_idx, required_cols]
        .sort_values(["shared_param_set", "horizon"])
        .reset_index(drop=True)
    )

    # Average across the best horizons for each shared parameter set
    avg_by_set = (
        best_per_set_horizon
        .groupby("shared_param_set", as_index=False)
        .agg(
            avg_before_val_real_mae=("before_val_real_mae", "mean"),
            avg_after_val_real_mae=("after_val_real_mae", "mean"),
            avg_before_test_real_mae=("before_test_real_mae", "mean"),
            avg_after_test_real_mae=("after_test_real_mae", "mean"),
            horizons_found=("horizon", "nunique"),
        )
        .sort_values("avg_after_test_real_mae")
        .reset_index(drop=True)
    )


    # =========================
    # Save structured tables
    # =========================

    best_per_set_horizon.to_csv(output_csv_path, index=False)
    avg_by_set.to_csv(avg_csv_path, index=False)

    print(f"Saved best per set/horizon CSV to: {output_csv_path}")
    print(f"Saved average by set CSV to: {avg_csv_path}")


    # =========================
    # Print each parameter set separately with AVG row
    # =========================

    display_cols = [
        "horizon",
        "candidate_index",
        "tuning_batch_size",
        "best_epoch",
        "before_val_real_mae",
        "after_val_real_mae",
        "before_test_real_mae",
        "after_test_real_mae",
    ]

    metric_cols = [
        "before_val_real_mae",
        "after_val_real_mae",
        "before_test_real_mae",
        "after_test_real_mae",
    ]

    # Sort parameter sets by best average after_test_real_mae first
    set_order = avg_by_set["shared_param_set"].tolist()

    all_output_blocks = []

    for rank, shared_set in enumerate(set_order, start=1):
        group = (
            best_per_set_horizon[
                best_per_set_horizon["shared_param_set"] == shared_set
            ]
            .sort_values("horizon")
            .reset_index(drop=True)
        )

        params = group["shared_param_params"].iloc[0]
        horizons_found = group["horizon"].nunique()

        avg_before_val_real_mae = group["before_val_real_mae"].mean()
        avg_after_val_real_mae = group["after_val_real_mae"].mean()
        avg_before_test_real_mae = group["before_test_real_mae"].mean()
        avg_after_test_real_mae = group["after_test_real_mae"].mean()

        # Build table with an AVG row at the end
        table = group[display_cols].copy()

        avg_row = {
            "horizon": "AVG",
            "candidate_index": "",
            "tuning_batch_size": "",
            "best_epoch": "",
            "before_val_real_mae": avg_before_val_real_mae,
            "after_val_real_mae": avg_after_val_real_mae,
            "before_test_real_mae": avg_before_test_real_mae,
            "after_test_real_mae": avg_after_test_real_mae,
        }

        table = pd.concat(
            [table, pd.DataFrame([avg_row])],
            ignore_index=True
        )

        # Round numeric metric columns for cleaner display
        table[metric_cols] = table[metric_cols].round(6)

        block = []
        block.append("=" * 140)
        block.append(f"Rank {rank} | Shared parameter set {shared_set}")
        block.append(f"Average after_test_real_mae: {avg_after_test_real_mae:.6f}")
        block.append(f"Horizons found: {horizons_found}")
        block.append(params)
        block.append("-" * 140)
        block.append(table.to_string(index=False))
        block.append("")

        block_text = "\n".join(block)

        # Print each parameter set as one output block
        print(block_text)

        all_output_blocks.append(block_text)


    # =========================
    # Save the same grouped output to text file
    # =========================

    Path(output_txt_path).write_text(
        "\n".join(all_output_blocks),
        encoding="utf-8"
    )

    print(f"Saved grouped text output to: {output_txt_path}")


    # =========================
    # Print the overall best shared parameter set only
    # =========================

    best_shared_set = avg_by_set.iloc[0]["shared_param_set"]
    best_avg = avg_by_set.iloc[0]["avg_after_test_real_mae"]

    best_group = (
        best_per_set_horizon[
            best_per_set_horizon["shared_param_set"] == best_shared_set
        ]
        .sort_values("horizon")
        .reset_index(drop=True)
    )

    best_table = best_group[display_cols].copy()

    best_avg_row = {
        "horizon": "AVG",
        "candidate_index": "",
        "tuning_batch_size": "",
        "best_epoch": "",
        "before_val_real_mae": best_group["before_val_real_mae"].mean(),
        "after_val_real_mae": best_group["after_val_real_mae"].mean(),
        "before_test_real_mae": best_group["before_test_real_mae"].mean(),
        "after_test_real_mae": best_group["after_test_real_mae"].mean(),
    }

    best_table = pd.concat(
        [best_table, pd.DataFrame([best_avg_row])],
        ignore_index=True
    )

    best_table[metric_cols] = best_table[metric_cols].round(6)

    print("\n" + "#" * 140)
    print(f"Overall best shared parameter set: {best_shared_set}")
    print(f"Best average after_test_real_mae: {best_avg:.6f}")
    print("#" * 140)
    print(best_table.to_string(index=False))
    print("\nParams:")
    print(best_group["shared_param_params"].iloc[0])

### Agriculture

In [3]:
log_path = "/home/yuyan/tabdpt_mz/results/hpo/agriculture_v3_analyzed_time_features_random_search_20260426_025112/summary.log"  # Change this to your actual log file path
parse_results(log_path)

Parsed candidate rows: 3000
Shared parameter sets found: 100
Horizons found: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]
Saved best per set/horizon CSV to: best_after_test_real_mae_per_shared_param_set_horizon.csv
Saved average by set CSV to: avg_after_test_real_mae_by_shared_param_set.csv
Rank 1 | Shared parameter set 73
Average after_test_real_mae: 4.298179
Horizons found: 6
Used params | text_attn_layers=[13, 14, 15, 16] | epochs=100 | gate_lr=0.01 | text_attn_lr=0.001 | gate_logit_clamp=5.0 | max_context=16 | target_lag_count=24 | embedding_lag_count=3
--------------------------------------------------------------------------------------------------------------------------------------------
horizon candidate_index tuning_batch_size best_epoch  before_val_real_mae  after_val_real_mae  before_test_real_mae  after_test_real_mae
      1            2165               128          4             3.003203            2.957549              4.481608         

### Energy

In [4]:

log_path = "/home/yuyan/tabdpt_mz/results/hpo/energy_v3_analyzed_time_features_random_search_20260426_132533/summary.log"  # Change this to your actual log file path
parse_results(log_path)

Parsed candidate rows: 3000
Shared parameter sets found: 100
Horizons found: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]
Saved best per set/horizon CSV to: best_after_test_real_mae_per_shared_param_set_horizon.csv
Saved average by set CSV to: avg_after_test_real_mae_by_shared_param_set.csv
Rank 1 | Shared parameter set 96
Average after_test_real_mae: 0.040117
Horizons found: 6
Used params | text_attn_layers=[13, 14, 15, 16] | epochs=100 | gate_lr=0.05 | text_attn_lr=0.01 | gate_logit_clamp=5.0 | max_context=64 | target_lag_count=24 | embedding_lag_count=3
--------------------------------------------------------------------------------------------------------------------------------------------
horizon candidate_index tuning_batch_size best_epoch  before_val_real_mae  after_val_real_mae  before_test_real_mae  after_test_real_mae
      1            2852                16          9             0.010595            0.009420              0.015678          

### Economy

In [5]:

log_path = "/home/yuyan/tabdpt_mz/results/hpo/economy_v3_analyzed_time_features_random_search_20260426_094457/summary.log"  # Change this to your actual log file path
parse_results(log_path)

Parsed candidate rows: 3000
Shared parameter sets found: 100
Horizons found: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]
Saved best per set/horizon CSV to: best_after_test_real_mae_per_shared_param_set_horizon.csv
Saved average by set CSV to: avg_after_test_real_mae_by_shared_param_set.csv
Rank 1 | Shared parameter set 57
Average after_test_real_mae: 2001.120076
Horizons found: 6
Used params | text_attn_layers=[16] | epochs=100 | gate_lr=0.1 | text_attn_lr=0.003 | gate_logit_clamp=5.0 | max_context=32 | target_lag_count=24 | embedding_lag_count=3
--------------------------------------------------------------------------------------------------------------------------------------------
horizon candidate_index tuning_batch_size best_epoch  before_val_real_mae  after_val_real_mae  before_test_real_mae  after_test_real_mae
      1            1684                64          1          1860.234741         1827.975708           1554.048218          1544.9824

### Social goods

In [6]:
log_path = "/home/yuyan/tabdpt_mz/results/hpo/social_v3_analyzed_time_features_random_search copy_20260426_171154/summary.log"  # Change this to your actual log file path
parse_results(log_path)

Parsed candidate rows: 3000
Shared parameter sets found: 100
Horizons found: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]
Saved best per set/horizon CSV to: best_after_test_real_mae_per_shared_param_set_horizon.csv
Saved average by set CSV to: avg_after_test_real_mae_by_shared_param_set.csv
Rank 1 | Shared parameter set 28
Average after_test_real_mae: 0.693026
Horizons found: 6
Used params | text_attn_layers=[1, 2, 16] | epochs=100 | gate_lr=0.1 | text_attn_lr=0.01 | gate_logit_clamp=5.0 | max_context=64 | target_lag_count=24 | embedding_lag_count=3
--------------------------------------------------------------------------------------------------------------------------------------------
horizon candidate_index tuning_batch_size best_epoch  before_val_real_mae  after_val_real_mae  before_test_real_mae  after_test_real_mae
      1             814                64          6             0.139639            0.127875              0.362102             0.36

### Climate

In [7]:
log_path = "/home/yuyan/tabdpt_mz/results/hpo/climate_v3_analyzed_time_features_random_search_20260427_001914/summary.log"  # Change this to your actual log file path
parse_results(log_path)

Parsed candidate rows: 4500
Shared parameter sets found: 150
Horizons found: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]
Saved best per set/horizon CSV to: best_after_test_real_mae_per_shared_param_set_horizon.csv
Saved average by set CSV to: avg_after_test_real_mae_by_shared_param_set.csv
Rank 1 | Shared parameter set 78
Average after_test_real_mae: 5.655476
Horizons found: 6
Used params | text_attn_layers=[16] | epochs=100 | gate_lr=0.1 | text_attn_lr=0.0003 | gate_logit_clamp=5.0 | max_context=16 | target_lag_count=12 | embedding_lag_count=3
--------------------------------------------------------------------------------------------------------------------------------------------
horizon candidate_index tuning_batch_size best_epoch  before_val_real_mae  after_val_real_mae  before_test_real_mae  after_test_real_mae
      1            2311                 8          4             2.060023            1.895397              1.680875             1.495305